In [ ]:
%matplotlib tk
from equipment.equipment_init import *
from project.sc8583 import project

from time import sleep as delay
from interface.cui_logger import logger as log

if "ic" in globals():
    ic.close()

ic = project(device="sc8583", revision="1p1", emulator="cp2112", logging=False, is_gui=False)

In [ ]:
from phy.power_z import km003c
pz = km003c(port="COM16", logging=False)

In [ ]:
pz.pdo_list

In [ ]:
pz.select_pdo = 5

In [ ]:
ic.i2c_scan()

In [ ]:
ic.m3_init_ref(op_mode=2)

In [ ]:
ic.IIN_UCP_DIS = 1

In [ ]:
pz.cfg_all = 7.8, 2.75

In [ ]:
ic.ADC_EN = 1
ic.ADC_RATE = 1
ic.status_adc
ic.ADC_EN = 0

In [ ]:
ic.status

In [ ]:
ic.enable_vin_charging

In [ ]:
ic.disable_charging

In [ ]:
import numpy as np
vbus_list = list(np.arange(8.0, 9.0, 0.02))

print("pps_v    pz_v     pz_i     pz_p      iin_adc   vin_adc   p_adc      p_diff")

log.output_set_filename(title="[EP-TA865 65W] PD TA + Power-Z + SC8583")
log.output_csv(message=["PPS_V", "Power-Z Voltage", "Power-Z Current", "Power-Z Power", "SC8583 IIN_ADC", "SC8583 VIN_ADC", "SC8583 Power", "Power_Diff"])

for v_set in vbus_list:

    pz.cfg_all = round(v_set, 2), 2.75
    delay(0.1)

    read_vip = pz.read_all
    v_ret = round(read_vip[0], 3)
    i_ret = round(read_vip[1], 3)
    p_ret = round(read_vip[2], 3)
    vin   = ic.vin_adc
    iin   = ic.iin_adc
    p_adc = vin * iin

    log.output_csv(message=[round(v_set, 3), round(v_ret, 3), round(i_ret, 3), round(p_ret, 3), round(iin, 3), round(vin, 3), round(p_adc, 3), round(p_ret-p_adc, 3)])
    print(f"{round(v_set, 2):.02f}V    {v_ret:.03f}V   {i_ret:.03f}A   {p_ret:.03f}W   {iin:.03f}A    {vin:.03f}A    {p_adc:.03f}W    {p_ret-p_adc:.03f}W")

    if v_ret < 6:
        ic.disable_charging
        break

    delay(0.2)

pz.cfg_all = 8.1, 2.75

In [ ]:
c1 = "C1NA"
c2 = "C2NA"
c3 = "C1NB"
c4 = "C2NB"
ds.save_waveform = f"[SC8583 EVM] parallel resistor on C2A {c1} {c2} {c3} {c4} PIN_DIAG_FAIL 6p9kR"

In [ ]:
c1 = "C1PA"
c2 = "C1NA"
c3 = "C2PA"
c4 = "C2NA"

for n in range(1, 2):
    ds.single
    delay(1.5)
    ic.enable_vin_charging
    delay(1)
    print(f"switching status : {ic.CP_SWITCHING_STAT}")
    # ds.save_waveform = f"[SC8583 CFLY_SHORT] {c1} 4v offset, {c2}, {c3} 3.5v offset, {c4} - 1kh 140mV pkpk waveform #{n} (switching={ic.CP_SWITCHING_STAT})"
    ic.disable_charging
    delay(1)

In [ ]:
c1 = "C1NA"
c2 = "C2NA"
c3 = "C1NB"
c4 = "C2NB"

ds.ch1.vertical_scale_grid = 0.1, 1
ds.ch2.vertical_scale_grid = 0.1, -2
ds.ch3.vertical_scale_grid = 0.1, 0
ds.ch4.vertical_scale_grid = 0.1, -3
ds.ch1.bw_20MHz
ds.ch2.bw_20MHz
ds.ch3.bw_20MHz
ds.ch4.bw_20MHz

# ds.ch4.label = f"{c1}, {c2}, {c3}, {c4}"

ds.ch1.label = c1
ds.ch2.label = c2
ds.ch3.label = c3
ds.ch4.label = c4

In [ ]:
ds.save_waveform = f"[SC8583 EVM] parallel resistor on C2A {c1} {c2} {c3} {c4} CFLY_SHORT 5kR"

In [ ]:
ds.save_waveform = f"[SC8583 CFLY_SHORT] {c1} 4v offset, {c2}, {c3} 3.5v offset, {c4} - 500hz 120mV pkpk waveform #cursor 2"

In [ ]:
ic.status

In [ ]:
ic.ADC_EN = 1
ic.status_adc

In [ ]:
pz.init_powerz

pps_temp = 8.0
vout = 4.0

pz.cfg_all = pps_temp, 3
delay(1.5)
ret_vbus = pz.voltage
print(f"initial power-z voltage : {ret_vbus:.03f}")

while True:
    ret_vbus = pz.voltage
    if ret_vbus < vout*2.02:
        pps_temp += 0.02
        pz.cfg_all = pps_temp, 3
        print(f"PPS={pps_temp:.03f}  RET={ret_vbus:.03f}  DIFF={pps_temp-ret_vbus:.03f}")
        delay(0.2)
    else:
        print("break #1")
        break
    if pps_temp > 9.5:
        print("break #2")
        break

import numpy as np
vbus_list = list(np.arange(pps_temp, 9.51, 0.02))

In [ ]:
pz.init_powerz
pz.cfg_all = 8.3, 3

In [ ]:
ic.m3_init_ref(op_mode=2)
ic.IIN_UCP_DIS = 1
ic.ADC_EN = 1
ic.ADC_RATE = 0

In [ ]:
ic.enable_vin_charging

In [ ]:
log.output_set_filename(title="[MAXTIL] PD TA + Power-Z")
log.output_csv(message=["PPS (V)", "VIN (V)", "IIN (A)", "IIN_ADC (A)", "Diff (A)"])

for v in vbus_list:
    vbus = round(v, 3)
    pz.cfg_all = vbus, 3
    delay(0.5)
    cp_vbus = round(ic.vin_adc, 3)
    cp_iin = round(ic.iin_adc, 3)
    pz_iin = round(pz.current, 3)
    diff   = round((pz_iin - cp_iin), 3)
    print(f"VBUS={vbus:.03f}  VIN={cp_vbus:.03f}  Power-Z_IIN={pz_iin:.03f}  SC8583_IIN={cp_iin:.03f}  -->  Diff={pz_iin-cp_iin:.03f}")
    log.output_csv(message=[vbus, cp_vbus, cp_iin, pz_iin, diff])

    if cp_iin > 3: break

pz.cfg_all = pps_temp, 2

In [ ]:
iteration_vbus = [8.64, 8.66, 8.68, 8.7, 8.72, 8.74, 8.76, 8.78, 8.8, 8.82, 8.84, 8.86, 8.88, 8.9, 8.92, 8.94, 8.96, 8.98, 9, 9.02, 9.04, 9.06, 9.08, 9.1, 9.12, 9.14]
log.output_set_filename(title="[MAXTIL] PD TA + Power-Z, 500mA 100Hz load")
# log.output_csv(message=["N", "CP", "PPS (V)", "VIN (V)", "IIN (A)", "IIN_ADC (A)", "Diff (V)", "Diff (A)"])

for n in range(1, 2):

    for v in iteration_vbus:
        vbus = round(v, 3)
        pz.cfg_all = vbus, 3
        delay(0.15)
        cp_vbus = round(ic.vin_adc, 3)
        cp_iin = round(ic.iin_adc, 3)
        pz_iin = round(pz.current, 3)
        diff_v = round(vbus - cp_vbus, 3)
        diff_i   = round((pz_iin - cp_iin), 3)
        sw_stat = ic.CP_SWITCHING_STAT
        print(f"[{n}] CP={sw_stat} VBUS={vbus:.03f}  VIN={cp_vbus:.03f}  Power-Z_IIN={pz_iin:.03f}  SC8583_IIN={cp_iin:.03f}  -->  Diff={pz_iin-cp_iin:.03f}")
        log.output_csv(message=[n, sw_stat, vbus, cp_vbus, cp_iin, pz_iin, diff_v, diff_i])

        if cp_iin > 3: break
        if sw_stat != 1: break

    pz.cfg_all = iteration_vbus[0], 2
    delay(1)
    print(f"------------------------------------------------------------------------------------")

In [ ]:
import numpy as np
# iteration_vbus = [8.64, 8.66, 8.68, 8.7, 8.72, 8.74, 8.76, 8.78, 8.8, 8.82, 8.84, 8.86, 8.88, 8.9, 8.92, 8.94, 8.96, 8.98, 9, 9.02, 9.04, 9.06, 9.08, 9.1, 9.12, 9.14]
iteration_vbus = list(np.arange(8.5, 9.301, 0.04))
log.output_set_filename(title="[MAXTIL] PD TA + Power-Z, 100mA 100Hz 25p 200ms delay load")
log.output_csv(message=["N", "CP", "PPS (V)", "VIN (V)", "IIN_ADC (A)"])

for n in range(101):

    for v in iteration_vbus:
        vbus = round(v, 3)
        pz.cfg_all = vbus, 3
        delay(0.2)
        cp_vbus = round(ic.vin_adc, 3)
        cp_iin = round(ic.iin_adc, 3)
        # pz_iin = round(pz.current, 3)
        # diff_v = round(vbus - cp_vbus, 3)
        # diff_i   = round((pz_iin - cp_iin), 3)
        sw_stat = ic.CP_SWITCHING_STAT
        print(f"[{n}] PPS={vbus:.03f}  CP={sw_stat}  VIN={cp_vbus:.03f}  SC8583_IIN={cp_iin:.03f}")
        log.output_csv(message=[n, sw_stat, vbus, cp_vbus, cp_iin])

        if cp_iin > 3: break
        if sw_stat != 1: break

    pz.cfg_all = iteration_vbus[0], 2
    delay(1)
    print(f"-------------------------------------")

In [ ]:
ic.disable_charging

In [ ]:
pz.cfg_all = 8.6, 3

In [ ]:
ic.status

In [ ]:
sweep_list = list()
for v in range(1, 51, 5):
    sweep_list.append(v/10)

In [ ]:
dm.set_nplc = 1
ps.ch2.cfg_all = 0.01, 0.01
ps.ch2.enable
dm.current_1E_3 * 1e+6

In [ ]:
ps.ch2.cfg_all = 0.05, 0.01
# bs.vset = 0.05

for voltage in sweep_list:
    # bs.vset = voltage - 0.01
    ps.ch2.vset = voltage
    delay(0.3)
    dm.current_100E_3
    i_ret = dm.current_1E_3 * 1e+6
    print(f"{voltage:.01f} {i_ret:.06f}")
    if i_ret > 5000: break

ps.ch2.cfg_all = 0.05, 0.01
# bs.vset = 0.05

In [ ]:
import time
ps.ch2.cfg_all = 0.01, 0.01
ps.ch2.enable
start = time.time()
ps.ch2.cfg_all = 5, 0.01
for n in range(1, 21):
    i_ret = dm.current_1E_3 * 1e+6
    lap_time = time.time() - start
    print(f"{lap_time:7.01f} {i_ret:.06f}")
ps.ch2.cfg_all = 0.01, 0.01

In [ ]:
dm.current_10E_6

In [ ]:
pz.init_powerz
pz.pdo_list

In [ ]:
import numpy as np
vbus_list = list(np.arange(8.5, 9.12, 0.02))

In [ ]:
pz.cfg_all = 8.2, 2.75

In [ ]:
ic.ADC_EN = 1
ic.ADC_RATE = 0

In [ ]:
import numpy as np
vbus_list = list(np.arange(8.5, 9.12, 0.02))

print("pps_v    pz_v     pz_i     pz_p      iin_adc   vin_adc   p_adc      p_diff")

for v_set in vbus_list:

    pz.cfg_all = round(v_set, 2), 2.75
    delay(0.1)

    read_vip = pz.read_all
    v_ret = round(read_vip[0], 3)
    i_ret = round(read_vip[1], 3)
    p_ret = round(read_vip[2], 3)
    vin   = ic.vin_adc
    iin   = ic.iin_adc
    p_adc = vin * iin

    print(f"{round(v_set, 2):.02f}V    {v_ret:.03f}V   {i_ret:.03f}A   {p_ret:.03f}W   {iin:.03f}A    {vin:.03f}A    {p_adc:.03f}W    {p_ret-p_adc:.03f}W")
    delay(0.2)

pz.cfg_all = 8.2, 2.8

In [ ]:
pz.pdo_list

In [ ]:
ic.status

In [ ]:
ic.IIN_UCP_DIS = 1

In [ ]:
ds.ch1.vertical_scale_grid = 0.1, 2
ds.ch2.vertical_scale_grid = 0.1, 1
ds.ch3.vertical_scale_grid = 0.1, -2
ds.ch4.vertical_scale_grid = 0.1, -3

In [ ]:
ds.ch1.label = "C1NA"
ds.ch2.label = "C1NB"
ds.ch3.label = "C2NA"
ds.ch4.label = "C2NB"

In [ ]:
ds.ch1.bw_20MHz
ds.ch2.bw_20MHz
ds.ch3.bw_20MHz
ds.ch4.bw_20MHz

In [ ]:
ds.ch3.trigger_fall = 0.3

In [ ]:
ds.ch1.trigger_rise = 0.35

In [ ]:
ds.save_waveform = "[M3] CH1=C1NA CH2=C1NB CH3=C2NA CH4=C2NB #18"

In [ ]:
pz.pdo_list

In [ ]:
pz.select_pdo = 5

In [ ]:
pz.cfg_all = 9.5, 2.75

In [ ]:
ld.enable

In [ ]:
ld.cc.iset = 2.95

In [ ]:
ld.cc.iset = 3.015

In [ ]:
ld.disable